# dt / Coefficient-Threshold Sweep

This notebook follows the ground-reference correlator workflow from `eigenvectors.ipynb`, but sweeps over Trotter step size and coefficient cutoff.

- Columns: `dt = 0.01, 0.02, ..., 0.08`
- Rows: `CoeffTruncation` threshold from `1e-3` to `1e-5`
- Each panel shows only the dense exact curve and the coefficient-truncated Pauli-propagation curve.
- `T_max` is held fixed. If a `dt` does not divide `T_max`, the last step is shortened so every curve ends at the same final time.


In [ ]:
using Pkg
Pkg.activate("../")
Pkg.instantiate()


In [ ]:
using PauliOperators
using Plots
using LinearAlgebra
using Printf
using Plots.PlotMeasures


## Hamiltonian and Reference Helpers

In [ ]:
function heisenberg_xxx(N::Int; J::Real=1.0)
    H = PauliSum(N, ComplexF64)
    for i in 1:(N - 1)
        H[PauliBasis(Pauli(N; X=[i, i + 1]))] = J + 0im
        H[PauliBasis(Pauli(N; Y=[i, i + 1]))] = J + 0im
        H[PauliBasis(Pauli(N; Z=[i, i + 1]))] = 2J + 0im
    end
    return H
end

function ket_vector(psi::Ket{N}) where {N}
    v = zeros(ComplexF64, Int(2^N))
    v[Int(psi.v) + 1] = 1.0
    return v
end

function ground_reference_kets(H::PauliSum{N}; atol=1e-10, dominance=0.99) where {N}
    Hm = Hermitian(Matrix(H))
    vals, vecs = eigen(Hm)
    E0 = minimum(vals)
    ground_cols = findall(lambda -> isapprox(lambda, E0; atol=atol, rtol=0), vals)
    ground_weights = vec(sum(abs.(vecs[:, ground_cols]) .^ 2; dims=2))
    refs = Ket{N}.(Int128.(findall(w -> w >= dominance, ground_weights) .- 1))
    return E0, refs, ground_weights
end

function dominant_ground_reference(H::PauliSum{N}; atol=1e-10, dominance=0.10) where {N}
    E0, refs, ground_weights = ground_reference_kets(H; atol=atol, dominance=dominance)
    if isempty(refs)
        idx = argmax(ground_weights)
        error("No computational-basis state has ground-subspace weight >= $dominance. Best candidate is |$(idx - 1)> with weight $(round(ground_weights[idx]; digits=6)).")
    end
    return E0, refs[1], refs, ground_weights
end


## Exact and Trotter Curves

The exact eigensystem is cached once. The coefficient-truncated curve uses a fixed final time even when `dt` does not divide `T_max` exactly.

In [ ]:
function exact_two_point_setup(H::PauliSum{N}, Oi::PauliSum{N}, Oj::PauliSum{N}, psi::Ket{N}) where {N}
    Hm = Hermitian(Matrix(H))
    Oim = Matrix(Oi)
    Ojm = Matrix(Oj)

    psiv = ket_vector(psi)
    phiv = Ojm * psiv

    F = eigen(Hm)
    V = F.vectors
    lambda = F.values

    return (
        lambda=lambda,
        V=V,
        Oim=Oim,
        cpsi=V' * psiv,
        cphi=V' * phiv,
    )
end

function exact_two_point_curve(exact_data, times::AbstractVector)
    lambda = exact_data.lambda
    V = exact_data.V
    Oim = exact_data.Oim
    cpsi = exact_data.cpsi
    cphi = exact_data.cphi

    ev = zeros(ComplexF64, length(times))
    for (idx, t) in enumerate(times)
        psit = V * (cis.(-t .* lambda) .* cpsi)
        phit = V * (cis.(-t .* lambda) .* cphi)
        ev[idx] = psit' * Oim * phit
    end
    return ev
end

function fixed_final_time_grid(dt::Real, T_max::Real; atol=1e-12)
    n_full = floor(Int, (T_max + atol) / dt)
    times = collect((0:n_full) .* dt)
    if times[end] < T_max - atol
        push!(times, T_max)
    else
        times[end] = T_max
    end
    return times
end

function trotter_coeff_curve_fixed_T(H::PauliSum{N,T}, Oi::PauliSum{N,T}, Oj::PauliSum{N,T},
                                     psi::Ket{N}, times::AbstractVector, dt::Real,
                                     threshold::Real) where {N,T}
    truncation = CoeffTruncation(threshold)
    base_generators, base_angles = trotterize(H, dt; n_trotter=1, order=2)
    Ot = deepcopy(Oi)

    ev = zeros(ComplexF64, length(times))
    n_terms = zeros(Int, length(times))

    ev[1] = expectation_value(Ot * Oj, psi)
    n_terms[1] = length(Ot)

    for step in 2:length(times)
        step_dt = times[step] - times[step - 1]
        if isapprox(step_dt, dt; atol=1e-12, rtol=1e-12)
            Ot = evolve(Ot, base_generators, base_angles; truncation=truncation)
        else
            generators, angles = trotterize(H, step_dt; n_trotter=1, order=2)
            Ot = evolve(Ot, generators, angles; truncation=truncation)
        end
        ev[step] = expectation_value(Ot * Oj, psi)
        n_terms[step] = length(Ot)
    end

    return ev, n_terms
end


## Sweep Configuration

In [ ]:
N = 12
J = -1.0
H = heisenberg_xxx(N; J=J)

E0, psi_ref, ground_refs, ground_weights = dominant_ground_reference(H; dominance=0.10)

i = 4
j = 5
Xi = PauliSum(Pauli(N; X=[i]))
Xj = PauliSum(Pauli(N; X=[j]))

T_max = 5.0
dts = collect(0.01:0.01:0.08)
thresholds = collect(10.0 .^ range(-3, -5; length=8))

println("Ground-state energy E0 = $(round(E0; digits=8))")
println("Using psi_ref = |$(psi_ref.v)>")
println("Available product ground references: ", [Int(ref.v) for ref in ground_refs])
println("Sweeping $(length(dts)) dt values and $(length(thresholds)) coefficient thresholds.")


## Run the Live 8 x 8 Sweep

This cell updates the grid as each `(threshold, dt)` pair finishes. It also saves snapshots to `examples/dt_threshold_grid_frames/`, so you can open the newest PNG while the sweep is still running.

In [ ]:
threshold_label(x) = @sprintf("%.1e", x)

function clear_live_output()
    try
        if isdefined(Main, :IJulia)
            getproperty(Main, :IJulia).clear_output(true)
        end
    catch
        nothing
    end
    return nothing
end

function padded_limits(values; pad_fraction=0.08, fallback=(-1.05, 1.05))
    finite_values = collect(skipmissing(values[isfinite.(values)]))
    isempty(finite_values) && return fallback
    lo = minimum(finite_values)
    hi = maximum(finite_values)
    if isapprox(lo, hi; atol=1e-12, rtol=1e-12)
        pad = max(abs(lo), 1.0) * 0.08
    else
        pad = (hi - lo) * pad_fraction
    end
    return (lo - pad, hi + pad)
end

function component_limits(state; component=real)
    values = Float64[]
    for exact in state.exact_by_dt
        append!(values, component.(exact))
    end
    for coeff in state.coeff_curves
        coeff === nothing && continue
        append!(values, component.(coeff))
    end
    return padded_limits(values)
end

function make_live_grid(state; component=real, component_label="Re")
    n_thresholds = length(state.thresholds)
    n_dts = length(state.dts)
    ylims = component_limits(state; component=component)
    panels = Any[]

    for row in 1:n_thresholds
        for col in 1:n_dts
            times = state.times_by_dt[col]
            exact = state.exact_by_dt[col]
            coeff = state.coeff_curves[row, col]
            err = state.max_errors[row, col]

            row_label = "eps=$(threshold_label(state.thresholds[row]))"

            p = plot(
                times, component.(exact);
                label=(row == 1 && col == 1 ? "exact" : false),
                color=:black, lw=1.8,
                title=(row == 1 ? @sprintf("dt=%.2f", state.dts[col]) : ""),
                ylabel=(col == 1 ? "Coeff threshold\n$row_label" : ""),
                xlabel=(row == n_thresholds ? "t" : ""),
                legend=(row == 1 && col == 1 ? :topright : false),
                framestyle=:box,
                foreground_color_border=:black,
                gridalpha=0.18,
                margin=1.8Plots.mm,
                left_margin=(col == 1 ? 9Plots.mm : 2Plots.mm),
                tickfontsize=6,
                guidefontsize=8,
                titlefontsize=8,
                legendfontsize=6,
                xticks=(row == n_thresholds ? :auto : false),
                yticks=(col == 1 ? :auto : false),
                xlims=(0, state.T_max),
                ylims=ylims,
            )

            annotate!(p, state.T_max * 0.18, ylims[2] - 0.10 * (ylims[2] - ylims[1]),
                text(row_label, :black, 8, :left))


            if coeff === nothing
                annotate!(p, state.T_max / 2, sum(ylims) / 2,
                    text("pending", :gray, 7, :center))
            else
                plot!(p, times, component.(coeff);
                    label=(row == 1 && col == 1 ? "CoeffTruncation" : false),
                    color=:blue, lw=1.6, ls=:dash,
                )
                annotate!(p, state.T_max * 0.52, ylims[1] + 0.10 * (ylims[2] - ylims[1]),
                    text(@sprintf("err %.1e", err), :darkgray, 6, :center))
            end

            push!(panels, p)
        end
    end

    return plot(
        panels...;
        layout=(n_thresholds, n_dts),
        size=(2600, 2400),
        dpi=180,
        plot_title="$component_label C_$(i)$(j)(t): exact vs CoeffTruncation",
        plot_titlefontsize=18,
        margin=4Plots.mm,
        background_color=:white,
    )
end

function run_dt_threshold_sweep_live(H, Oi, Oj, psi, dts, thresholds, T_max;
                                     component=real, component_label="Re",
                                     output_dir="dt_threshold_grid_frames",
                                     display_updates=true,
                                     save_snapshots=true,
                                     save_every=1)
    mkpath(output_dir)
    exact_data = exact_two_point_setup(H, Oi, Oj, psi)

    times_by_dt = [fixed_final_time_grid(dt, T_max) for dt in dts]
    exact_by_dt = [exact_two_point_curve(exact_data, times) for times in times_by_dt]

    coeff_curves = Matrix{Union{Nothing,Vector{ComplexF64}}}(undef, length(thresholds), length(dts))
    term_counts = Matrix{Union{Nothing,Vector{Int}}}(undef, length(thresholds), length(dts))
    fill!(coeff_curves, nothing)
    fill!(term_counts, nothing)
    max_errors = fill(NaN, length(thresholds), length(dts))

    state = (
        dts=dts,
        thresholds=thresholds,
        T_max=T_max,
        times_by_dt=times_by_dt,
        exact_by_dt=exact_by_dt,
        coeff_curves=coeff_curves,
        term_counts=term_counts,
        max_errors=max_errors,
        output_dir=output_dir,
    )

    frame = 0
    total = length(dts) * length(thresholds)
    grid = make_live_grid(state; component=component, component_label=component_label)
    display_updates && display(grid)
    save_snapshots && savefig(grid, joinpath(output_dir, "frame_000_initial.png"))

    for (row, threshold) in enumerate(thresholds)
        for (col, dt) in enumerate(dts)
            times = times_by_dt[col]
            exact = exact_by_dt[col]
            @printf("[%02d/%02d] dt = %.2f, threshold = %.2e, steps = %d\n",
                    frame + 1, total, dt, threshold, length(times) - 1)
            flush(stdout)

            ev, nt = trotter_coeff_curve_fixed_T(H, Oi, Oj, psi, times, dt, threshold)
            coeff_curves[row, col] = ev
            term_counts[row, col] = nt
            max_errors[row, col] = maximum(abs.(ev .- exact))

            frame += 1
            @printf("    max error = %.3e, final terms = %d\n", max_errors[row, col], nt[end])
            flush(stdout)

            should_save = save_snapshots && (frame % save_every == 0 || frame == total)
            if display_updates || should_save
                grid = make_live_grid(state; component=component, component_label=component_label)
                if display_updates
                    clear_live_output()
                    display(grid)
                end
                if should_save
                    savefig(grid, joinpath(output_dir, @sprintf("frame_%03d.png", frame)))
                end
            end
        end
    end

    final_grid = make_live_grid(state; component=component, component_label=component_label)
    final_path = joinpath(output_dir, "dt_threshold_coeff_grid_$(lowercase(component_label)).png")
    savefig(final_grid, final_path)
    display_updates && display(final_grid)
    println("Saved final grid to: $final_path")

    return merge(state, (final_grid=final_grid, final_path=final_path))
end

@time results = run_dt_threshold_sweep_live(
    H, Xi, Xj, psi_ref, dts, thresholds, T_max;
    component=real,
    component_label="Re",
    output_dir="dt_threshold_grid_frames",
    display_updates=true,
    save_snapshots=true,
    save_every=1,
);


## Final 8 x 8 Exact-vs-CoeffTruncation Figure

In [ ]:
display(results.final_grid)
println("Final saved figure: ", results.final_path)


Optional: plot the imaginary part with the same sweep results.

In [ ]:
# grid_imag = make_live_grid(results; component=imag, component_label="Im")
# display(grid_imag)
# savefig(grid_imag, joinpath(results.output_dir, "dt_threshold_coeff_grid_im.png"))


## Error Matrix

Rows match the threshold axis and columns match the dt axis.

In [ ]:
println("dt columns: ", results.dts)
println("threshold rows: ", threshold_label.(results.thresholds))
display(results.max_errors)
